In [13]:
import pandas as pd
import numpy as np

# 1. Load the raw transactional dataset
print("Step 1: Loading raw dataset...")
df = pd.read_csv('Coffee Shop Sales.csv')
print(f"Loaded {len(df)} transactions.")

Step 1: Loading raw dataset...
Loaded 149116 transactions.


In [14]:
# 2. Time-Based Feature Engineering
print("\nStep 2: Processing time and date features...")
df['transaction_date'] = pd.to_datetime(df['transaction_date'])
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.day_name()
df['is_weekend'] = df['transaction_date'].dt.dayofweek.apply(lambda x: 1 if x >= 5 else 0)
df['hour'] = pd.to_datetime(df['transaction_time'], format='%H:%M:%S').dt.hour

def get_shift(hour):
    if 6 <= hour < 12: return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    else: return 'Evening'

df['shift'] = df['hour'].apply(get_shift)
print("Done. Shift and time features created.")


Step 2: Processing time and date features...
Done. Shift and time features created.


In [15]:
# 3. Create the Category Pivot (The Core Logic)
print("\nStep 3: Pivoting data to calculate shift totals for all 9 categories...")
# This turns the 'product_category' rows into 9 individual columns
category_pivot = df.pivot_table(
    index=['transaction_date', 'store_location', 'month', 'day_of_week', 'is_weekend', 'shift'],
    columns='product_category',
    values='transaction_qty',
    aggfunc='sum',
    fill_value=0
).reset_index()

# Clean up column names (replacing spaces with underscores for code compatibility)
category_pivot.columns = [c.replace(' ', '_') for c in category_pivot.columns]
print("Done. All 9 categories transformed into target columns.")


Step 3: Pivoting data to calculate shift totals for all 9 categories...
Done. All 9 categories transformed into target columns.


In [16]:
# 4. Adding Context (Total Shift Volume)
print("\nStep 4: Calculating total shift volume as an anchor feature...")
# Define the 9 target categories specifically
target_categories = [
    'Bakery', 'Branded', 'Coffee', 'Coffee_beans', 
    'Drinking_Chocolate', 'Flavours', 'Loose_Tea', 
    'Packaged_Chocolate', 'Tea'
]

# Create a total cups column to help the model understand the overall shop traffic
category_pivot['total_shift_cups'] = category_pivot[target_categories].sum(axis=1)


Step 4: Calculating total shift volume as an anchor feature...


In [17]:
# 5. Export the Final Training Set
print("\nStep 5: Exporting 'Model3_Category_Inventory.csv'...")
category_pivot.to_csv('Model3_Category_Inventory.csv', index=False)
print("Done! File is ready for Multi-Output Training.")

print("\n" + "="*40)
print("CATEGORIES TRACKED IN THIS MODEL:")
for i, cat in enumerate(target_categories, 1):
    print(f"{i}. {cat}")
print("="*40)
print(category_pivot[['transaction_date', 'shift', 'Coffee', 'Tea', 'Bakery']].head())


Step 5: Exporting 'Model3_Category_Inventory.csv'...
Done! File is ready for Multi-Output Training.

CATEGORIES TRACKED IN THIS MODEL:
1. Bakery
2. Branded
3. Coffee
4. Coffee_beans
5. Drinking_Chocolate
6. Flavours
7. Loose_Tea
8. Packaged_Chocolate
9. Tea
  transaction_date      shift  Coffee  Tea  Bakery
0       2023-01-01  Afternoon      74   61       7
1       2023-01-01    Evening      38   28       7
2       2023-01-01    Morning      13   10       3
3       2023-01-01  Afternoon      55   41       6
4       2023-01-01    Evening      40   34       7


In [12]:
# 6. Export the new Pivot Dataset
print("\nStep 4: Exporting Model3_Category_Inventory.csv...")
category_pivot.to_csv('Model3_Category_Inventory.csv', index=False)
print("Done! This dataset is now optimized for inventory forecasting.")

print("\n" + "="*40)
print("PREVIEW OF NEW DATA STRUCTURE:")
print("="*40)
print(category_pivot[['transaction_date', 'shift', 'Coffee', 'Tea', 'Bakery']].head())


Step 4: Exporting Model3_Category_Inventory.csv...
Done! This dataset is now optimized for inventory forecasting.

PREVIEW OF NEW DATA STRUCTURE:
  transaction_date      shift  Coffee  Tea  Bakery
0       2023-01-01  Afternoon      74   61       7
1       2023-01-01    Evening      38   28       7
2       2023-01-01    Morning      13   10       3
3       2023-01-01  Afternoon      55   41       6
4       2023-01-01    Evening      40   34       7
